# 01 Exploration

What the simulated data looks like before any models ever sees it. Every figure is
saved to `results/figures/`.

Each row is one **candidate cell at a forced guess**: its 9×9 window of visible
cell states, the oracle's exact mine probability `q`, and whether it really is a
mine. A **position** is one forced guess (one game, one guess index) and holds
all of its candidate rows.

In [ ]:
%load_ext autoreload
%autoreload 2

import glob

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from scipy.stats import binom

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from inf_minesweeper.board import FLAGGED, HIDDEN, MINE_HIT, WRONG_FLAG
from inf_minesweeper.loader import load, split_by_game
from inf_minesweeper.features import FEATURE_NAMES, handcrafted, one_hot, symmetries
from inf_minesweeper.metrics import reliability
from inf_minesweeper.plotting import DIV, MUTED, SEQ, SERIES, no_grid, save, use_style
from inf_minesweeper.screenshots import cleared_patterns

DATA = "../data/raw/guesses.npz"
SCREENSHOTS = "../data/screenshots/*.png"
FIGS = "../results/figures"
use_style()
rng = np.random.default_rng(0)

## Dataset Overview

In [ ]:
d = load(DATA)
W, q, y = d["window"], d["q"], d["mine"].astype(float)
game, position, chosen = d["game"], d["position"], d["chosen"]

# One integer ID per forced-guess position.
_, pos_id = np.unique(np.stack([game, position]), axis=1, return_inverse=True)
pos_id = pos_id.ravel()
n_pos = pos_id.max() + 1

print("config:", d["config"])
pd.Series({
    "games": len(d["game_seed"]),
    "forced-guess positions": n_pos,
    "rows (candidates)": len(q),
    "candidates per position": round(len(q) / n_pos, 1),
    "mean oracle q": round(q.mean(), 4),
    "actual mine rate": round(y.mean(), 4),
}).to_frame("value")

## How oftan games reach a forced guess

Left: forced guesses per game. Right: candidate cells per forced guess, which is how many choices the `guesser` has each time.

In [ ]:
guesses = d["game_guesses"]
cands = np.bincount(pos_id)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
ax = axes[0]
ax.hist(guesses, bins=np.arange(guesses.max() + 2) - 0.5, color=SERIES[0])
ax.set(title="Forced guesses per game", xlabel="candidate cells", ylabel="positions")
ax = axes[1]
ax.hist(cands, bins=40, color=SERIES[0], rwidth=0.85)
ax.set(title="Candidates per forced guess", xlabel="candidate cells", ylabel="positions")
fig.tight_layout()
save(fig, "01_guesses_and_candidates", FIGS)
print(f"median {np.median(guesses):.0f} guesses per game, "
      f"median {np.median(cands):.0f} candidates per position")

## Oracle probabilities

Left: how the oracle's probabilities are spread across the rows. Right: oracle probability checked against true density. Each point is a bin of rows; on the diagonal means a predicted 30% really is a mine 30% of the time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axes[0]
ax.hist(q, bins=50, range=(0, 1), color=SERIES[0], rwidth=0.9)
ax.set(title="Oracle mine probability, all candidates", xlabel="q", ylabel="rows")

ax = axes[1]
pred, obs, n = reliability(y, q, bins=20)
ax.plot([0, 1], [0, 1], color=MUTED, lw=1)
ax.plot(pred, obs, "o-", color=SERIES[0], ms=5)
ax.set(title="Oracle calibration", xlabel="predicted (bin mean of q)",
       ylabel="observed mine rate", xlim=(0, 1), ylim=(0, 1), aspect="equal")
fig.tight_layout()
save(fig, "01_oracle_q", FIGS)
print(f"largest bin gap: {np.abs(pred - obs).max():.3f}")

## The best available guess

At each forced guess, the lowest `q` among the candidates is the risk even a perfect player has to take. The Minesweeper Wiki says 50/50s are the most common kind of forced guess; the share of positions whose best option is exactly 0.5 tests that for this board.

In [ ]:
best = np.full(n_pos, np.inf)
np.minimum.at(best, pos_id, q)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(best, bins=50, range=(0, 1), color=SERIES[0], rwidth=0.9)
ax.axvline(np.median(best), color=MUTED, lw=1)      #type:ignore
ax.annotate(f"median {np.median(best):.2f}", (np.median(best), 1), xycoords=("data", "axes fraction"),      #type:ignore
            xytext=(4, -2), textcoords="offset points", va="top", color=MUTED,
            bbox=dict(boxstyle="square,pad=0.15", fc="#fcfcfb", ec="none"))
ax.set(title="Safest candidate at each forced guess", xlabel="lowest q in the position",
       ylabel="positions")
save(fig, "01_best_guess", FIGS)

pd.Series({
    "best q is exactly 0.5": np.mean(np.isclose(best, 0.5, atol=1e-6)),
    "best q is at most 0.10": np.mean(best <= 0.10),
    "best q is at most 0.25": np.mean(best <= 0.25),
    "best q is above 0.40": np.mean(best > 0.40),
    "mean best q (oracle hit value)": best.mean(),
}).round(3).to_frame("share of positions")

## What the windows contain

Left: share of windows where each position is still hidden; the centre is the candidate itself and is always hidden. Right: the mix of cell states by distance from the centre (rings 1 to 4). Far rings that are mostly hidden carry little information, which bounds how much a larger window can help.

In [ ]:
groups = {
    "hidden": W == HIDDEN,
    "revealed number": W <= 8,
    "flag or mine hit": (W == FLAGGED) | (W == MINE_HIT) | (W == WRONG_FLAG),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4.6))
for ax, (name, mask) in zip(axes, groups.items()):
    share = 100 * mask.mean(0).astype(float)
    share[4, 4] = np.nan                           # the candidate itself
    lo, hi = np.nanmin(share), np.nanmax(share)
    ax.imshow(share, cmap=SEQ, vmin=lo, vmax=hi)
    for (j, i), v in np.ndenumerate(share):
        if np.isnan(v):
            ax.text(i, j, "?", ha="center", va="center", fontsize=9, color=MUTED)
        else:
            dark = (v - lo) / (hi - lo) > 0.55
            ax.text(i, j, f"{v:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if dark else "#0b0b0b")
    ax.add_patch(Rectangle((2.5, 2.5), 3, 3, fill=False, ec="#0b0b0b", lw=1))
    ax.set(title=f"{name}, % of windows", xticks=[], yticks=[])
    no_grid(ax)
fig.tight_layout()
save(fig, "01_window_contents", FIGS);

In [ ]:
ring = np.maximum(*np.abs(np.mgrid[-4:5, -4:5]))     # Chebyshev distance from centre
by_ring = pd.DataFrame(
    [{name: mask[:, ring == r].mean() for name, mask in groups.items()} for r in range(1, 5)],
    index=range(1, 5))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for (name, col), color in zip(by_ring.items(), SERIES):
    ax.plot(by_ring.index, col, "o-", color=color, ms=6)
    ax.annotate(name, (4, col.iloc[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", color=color)
ax.set(title="Cell states by distance from the candidate", xlabel="ring",
       ylabel="share of cells", xticks=range(1, 5), xlim=(0.8, 5.6), ylim=(0, 1))
save(fig, "01_window_rings", FIGS)
by_ring.round(3)

## Which features hold signifance

Mean oracle probability against each handcrafted feature (rows binned by feature). A flat line means the features says little on its own.

In [ ]:
F = pd.DataFrame(handcrafted(W), columns=FEATURE_NAMES)
F["q"] = q
show = ["n_numbers", "n_unknown", "sum_numbers", "ratio_mean", "ratio_max", "ratio_union"]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=True)
for ax, name in zip(axes.flat, show):
    x = F[name]
    if x.nunique() <= 25:                                   # counts: one point per value
        g = F.groupby(name)["q"].agg(["mean", "size"])
    else:                                                   # ratios: 20 equal-count bins
        g = F.groupby(pd.qcut(x, 20, duplicates="drop"), observed=True)["q"].agg(["mean", "size"])
        g.index = F.groupby(pd.qcut(x, 20, duplicates="drop"), observed=True)[name].mean().values
    g = g[g["size"] >= 50]
    ax.plot(g.index, g["mean"], "o-", color=SERIES[0], ms=4)
    ax.set(title=name, ylim=(0, 1))
axes[0].set_ylabel("mean oracle q")
fig.tight_layout()
save(fig, "01_q_by_feature", FIGS)
F[FEATURE_NAMES + ["q"]].corr()["q"].drop("q").round(3).to_frame("correlation w/ q")

## Symmetry checks

The simulator should have no preferred direction. Rows from one game are correlated, so noise is measured by resampling whole games (a game bootstrap) rather than treating rows as independent. A deviation of more than about 3 bootstrap standard deviations $(|z| > 3)$ would point to a real bias.

Left: mine rate by the candidate's position inside its 8×8 sector, minus the overall rate. Below: the average share-hidden window against its mirror images.

In [ ]:
mine = y == 1
hidden = W == HIDDEN
is_num = W <= 8

ug, gi = np.unique(game, return_inverse=True)
gi = gi.ravel()
G = len(ug)
boot_w = np.random.default_rng(0).poisson(1.0, size=(200, G))   # game bootstrap weights

def boot_means(sums, n):
    """Means from per-game sums (G x k) and counts (G or G x k): the full-data
    value and 200 bootstrap replicates that resample whole games."""
    n = n if n.ndim == 2 else n[:, None]
    return sums.sum(0) / n.sum(0), (boot_w @ sums) / (boot_w @ n)

# mine rate by position in sector (64 cells), as rate minus overall
cell = ((d["y"] % 8) * 8 + d["x"] % 8).astype(np.int64)
n_cell = np.bincount(gi * 64 + cell, minlength=G * 64).reshape(G, 64).astype(float)
m_cell = np.bincount(gi * 64 + cell, weights=y, minlength=G * 64).reshape(G, 64)
rate, boot_rate = boot_means(m_cell, n_cell)
dev = (rate - y.mean()).reshape(8, 8)
z_sector = dev / boot_rate.std(0).reshape(8, 8)

fig, ax = plt.subplots(figsize=(5, 4.2))
lim = np.abs(dev).max()
im = ax.imshow(dev, cmap=DIV, vmin=-lim, vmax=lim)
ax.set(title="Mine rate by position in sector, minus overall", xlabel="x mod 8", ylabel="y mod 8")
no_grid(ax)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
save(fig, "01_symmetry_sector", FIGS)

# mirror images of the mean share-hidden window
h_sums = pd.DataFrame(hidden.reshape(len(y), 81)).groupby(gi).sum().to_numpy(float)
mean_h, boot_h = boot_means(h_sums, np.bincount(gi).astype(float))

def mirror(f):
    obs = mean_h.reshape(9, 9) - f(mean_h.reshape(9, 9))
    sd = np.stack([m.reshape(9, 9) - f(m.reshape(9, 9)) for m in boot_h]).std(0)
    z = np.abs(obs) / np.where(sd > 0, sd, np.inf)
    return np.abs(obs).max(), z.max()

rows = {"sector position (largest of 64)": (np.abs(dev).max(), np.abs(z_sector).max())}
for name, f in [("left-right mirror", lambda a: a[:, ::-1]),
                ("up-down mirror", lambda a: a[::-1, :]),
                ("transpose", lambda a: a.T)]:
    rows[name] = mirror(f)
pd.DataFrame(rows, index=["largest difference", "largest |z|"]).T.round(3)

## Game outcomes

In [ ]:
g = pd.DataFrame({k[5:]: v for k, v in d.items() if k.startswith("game_")})
if "sectors" not in g:                       # older simulate.py didn't store it
    g["sectors"] = (2 * d["config"]["radius"] - 1) ** 2
dead_end = (g["sectors"] - g["cleared"] - g["locked"]) > g["sectors"] / 2

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
axes[0].hist(g["score"], bins=40, color=SERIES[0], rwidth=0.9)
axes[0].set(title="Score per game", xlabel="score", ylabel="games")
axes[1].hist(g["cleared"], bins=np.arange(g["sectors"].max() + 2) - 0.5, color=SERIES[0], rwidth=0.85)
axes[1].set(title="Inner sectors cleared", xlabel=f"sectors (of {g['sectors'].iloc[0]})")
axes[2].hist(g["locked"], bins=np.arange(g["locked"].max() + 2) - 0.5, color=SERIES[1], rwidth=0.85)
axes[2].set(title="Inner sectors locked", xlabel="sectors")
fig.tight_layout()
save(fig, "01_game_outcomes", FIGS)

print(f"early dead ends (most inner sectors never resolved): {dead_end.mean():.1%} of games")
g[["score", "cleared", "locked", "guesses", "mines_hit"]].describe().round(2).T

## Train/test split

Splitting by whole games keeps correlated rows from one game on the same side. Both sides should look alike.

In [ ]:
train, test = split_by_game(game)
split = pd.DataFrame({
    side: {
        "games": len(np.unique(game[m])),
        "rows": int(m.sum()),
        "mine rate": y[m].mean(),
        "mean q": q[m].mean(),
        "median q": float(np.median(q[m])),
    }
    for side, m in [("train", train), ("test", test)]
})

fig, ax = plt.subplots(figsize=(6.5, 3.3))
for (name, m), color in zip([("train", train), ("test", test)], SERIES):
    ax.hist(q[m], bins=50, range=(0, 1), density=True, histtype="step", lw=2, color=color, label=name)
ax.legend()
ax.set(title="Oracle q on each side of the split", xlabel="q", ylabel="density")
save(fig, "01_split_check", FIGS)
split.round(4)

## Mine density from app screenshots

Mines per cleared sector, measured from screenshots by `screenshots.py`, against the binomial counts that independent placement at the measured density predicts.

In [ ]:
paths = sorted(glob.glob(SCREENSHOTS))
if not paths:
    print(f"no screenshots found at {SCREENSHOTS}; skipping")
else:
    counts = np.array([sum(p) for p in cleared_patterns(paths)])
    p_hat = counts.sum() / (64 * len(counts))
    k = np.arange(0, 31)

    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.hist(counts, bins=np.arange(32) - 0.5, color=SERIES[0], rwidth=0.85, label="cleared sectors")        #type:ignore
    ax.plot(k, binom.pmf(k, 64, p_hat) * len(counts), "o-", color=SERIES[1], ms=4,
            label=f"binomial, p = {p_hat:.3f}")
    ax.legend()
    ax.set(title="Mines per cleared sector in the app", xlabel="mines", ylabel="sectors",
           xlim=(0, 30))
    save(fig, "01_screenshot_density", FIGS)
    print(f"{len(counts)} sectors, density {p_hat:.3f}, variance {counts.var(ddof=1):.2f} "
          f"(binomial {64 * p_hat * (1 - p_hat):.2f})")

## Recurring local configurations

The same local shape can appear in 8 orientations (rotations and reflections), and raw clustering mostly sorts windows by which side the hidden area is on. So each 5×5 window is first turned to a standard orientation: the one that puts the most hidden cells toward the bottom right. Then, on a 30,000-row sample:

- **Left:** the first two principal components of the one-hot windows, colored by mean oracle `q`.
- **Right:** k-means clusters, each drawn as its share-hidden image with its size and mean `q`, sorted from safest to riskiest. These are the local shapes forced guesses come in.

In [ ]:
idx = rng.choice(len(q), size=min(30_000, len(q)), replace=False)
w5 = W[idx][:, 2:7, 2:7]
qs = q[idx]

# standard orientation: the symmetry that maximises hidden mass toward bottom right
syms = symmetries(w5)                                   # (8, N, 5, 5)
weight = np.add.outer(np.arange(5), np.arange(5)) * 1.0 + 0.01 * np.arange(25).reshape(5, 5)
score = ((syms == HIDDEN) * weight).sum(axis=(2, 3))    # (8, N)
w5c = syms[score.argmax(0), np.arange(len(idx))]

X = one_hot(np.pad(w5c, ((0, 0), (2, 2), (2, 2)), constant_values=HIDDEN), 5).astype(np.float32)
pca = PCA(n_components=10, random_state=0).fit(X)
Z = pca.transform(X)[:, :2]

K = 8
km = KMeans(n_clusters=K, n_init=4, random_state=0).fit(X)
lab = km.labels_
order = np.argsort([qs[lab == c].mean() for c in range(K)])

fig = plt.figure(figsize=(12, 3.6))
gs = fig.add_gridspec(2, K // 2 + 3, width_ratios=[1.4, 1.4, 0.15] + [1] * (K // 2))
ax = fig.add_subplot(gs[:, :2])
hb = ax.hexbin(Z[:, 0], Z[:, 1], C=qs, reduce_C_function=np.mean, gridsize=30,
               cmap=SEQ, vmin=0, vmax=1, mincnt=5)
ax.set(title="PCA, colored by mean q", xlabel="PC 1", ylabel="PC 2")
no_grid(ax)
fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.02)
for i, c in enumerate(order):
    ax = fig.add_subplot(gs[i // (K // 2), 3 + i % (K // 2)])
    ax.imshow((w5c[lab == c] == HIDDEN).mean(0), cmap=SEQ, vmin=0, vmax=1)
    ax.set_title(f"q {qs[lab == c].mean():.2f}, {np.mean(lab == c):.0%}", fontsize=9)
    ax.set(xticks=[], yticks=[])
    no_grid(ax)
fig.text(0.42, 0.98, "Clusters, safest to riskiest: share hidden (dark = usually hidden)",
         fontsize=10, va="top")
save(fig, "01_clusters", FIGS)

summary = pd.DataFrame({
    "share of rows": [np.mean(lab == c) for c in order],
    "mean q": [qs[lab == c].mean() for c in order],
    "mean hidden neighbors": [(w5c[lab == c][:, 1:4, 1:4] == HIDDEN).sum((1, 2)).mean() - 1 for c in order],
})
print(f"first two PCs explain {pca.explained_variance_ratio_[:2].sum():.0%} of the variance")
summary.round(3)